# 第11回：化学の知識を特徴量にする

**今日の問い：研究者の知識を、モデルへ渡せる形にするにはどうするか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。`DEEP DIVE`は経験者や自習向けの発展です。
分からないコードは、セル全体ではなく気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 化学的仮説を再計算可能な特徴量へ変え、交差検証でアブレーションする
- リークを避けたtarget encodingを、分割の内側で自作する
- 相互情報量・RFECVで特徴量を選び、適用領域の限界を意識する

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。
経験者は`CORE`を早めに終え、`DEEP DIVE`を5人で分担して読むと深まります。

### 先に押さえる言葉

- 特徴量設計：既存情報から予測に役立つ表現を作ること
- target encoding：カテゴリを目的変数の集約値で置き換える手法
- アブレーション：要素を足し引きして寄与を調べる比較
- RFECV：交差検証つきで再帰的に特徴量を削る選択法
- 適用領域：モデルが信頼できる入力範囲

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


## 特徴量設計＝あなたの化学知識をモデルへ渡す

ここは研究者の腕の見せどころです。モデルは与えられた列しか見ません。**「最適温度から離れるほど
収率が落ちる」**という知識を持っていても、`temperature_c`の生の値だけでは、モデルがその山型を
学ぶのは大変です。そこで、知識を**計算式**にして新しい列（特徴量）として渡します。これが特徴量設計です。

鉄則が2つあります。
1. **予測時点で計算できること**（第5回。実験後の値から作らない）。
2. **追加の効果は、同じ検証条件で前後比較して確かめる**（思い込みで良し悪しを決めない）。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


## TRY：仮説を計算式にする

2つの仮説を式にします。**「最適温度78℃からの距離」**（離れるほど収率減、という山型を直接表す）と、
**「単位時間あたりの濃度」**（濃度と時間の兼ね合い）。どちらも計画時に計算できる値です。


In [ ]:
engineered = df.copy()
engineered["temperature_distance"] = (engineered["temperature_c"] - 78).abs()
engineered["concentration_per_hour"] = engineered["concentration_m"] / engineered["reaction_time_h"]
engineered[["temperature_c", "temperature_distance", "concentration_per_hour"]].head()


### 読みどころ

`temperature_distance`は、78℃から上下どちらに離れても大きくなる値（絶対値）。第4回で見た「温度と収率の
山型」を、モデルにとって学びやすい**単調な形**に翻訳しています。生の温度より効くかどうかは、次で検証します。


## アブレーション：追加の効果を「同じ条件」で確かめる

**アブレーション**とは、要素を足し引きして寄与を測る比較のこと。特徴量を追加する前後で、
**同じモデル・同じ交差検証**でMAEを比べます。これをやらずに「良さそうだから採用」は禁物です。


In [ ]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

base = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
added = [*base, "temperature_distance", "concentration_per_hour"]
cv = KFold(5, shuffle=True, random_state=42)
for label, cols in {"追加前": base, "追加後": added}.items():
    est = make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42))
    scores = cross_val_score(est, engineered[cols], engineered["yield_pct"], cv=cv, scoring="neg_mean_absolute_error")
    print(f"{label}: MAE={-scores.mean():.3f} ± {scores.std():.3f}")


### 出力の読み方

- 「追加後」のMAEが「追加前」より**下がっていれば**、その特徴量は効いています。**ばらつき(±)より大きく**下がっているかも見ます（±の中の差は誤差かも）。
- 効かない・悪化することもあります。それも立派な結果——**「この仮説はこのモデルには効かなかった」**と分かるのが検証の価値です。悪化した実験も記録します（第12回）。


## CORE深掘り：関連の強い特徴量を選ぶ（相互情報量）

特徴量が増えると、効かない列がノイズになることも。**相互情報量（第4回）**で目的変数との関連が強い順に
並べ、上位k個を選びます。相関と違い、山型のような非線形の関連も拾えます。


In [ ]:
from sklearn.feature_selection import SelectKBest, mutual_info_regression

sel_data = engineered[added].fillna(engineered[added].median())
selector = SelectKBest(mutual_info_regression, k=4).fit(sel_data, engineered["yield_pct"])
pd.DataFrame({"特徴量": added, "MIスコア": selector.scores_, "選択": selector.get_support()}).sort_values("MIスコア", ascending=False).round(3)


### 出力の読み方

MIスコアの高い順に並び、上位4つに「選択=True」が付きます。自作した`temperature_distance`が上位に来て
いれば、狙いどおり効く特徴量を作れたということ。**化学の直感（山型）と数字が一致する**瞬間です。


## CHALLENGE：RDKitでSMILESから記述子を計算する

分子量やLogPは、本来は分子構造（SMILES）から計算できます。RDKitが入っていれば、エタノールの
記述子を実際に計算してみます。無い環境では自動でスキップし、計算済みの列で本編を進められます。


In [ ]:
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, Crippen
    molecule = Chem.MolFromSmiles("CCO")
    print("エタノールの分子量:", round(Descriptors.MolWt(molecule), 2))
    print("エタノールのLogP:", round(Crippen.MolLogP(molecule), 2))
except ImportError:
    print("RDKitは任意（uv sync --extra chemistry）。計算済みmolecular_weight/logp/tpsaで本編を進められます。")


### 読みどころ

RDKitが動けば、SMILES（`CCO`＝エタノール）から分子量やLogPが再現されます。**「記述子＝構造から計算できる
特徴量」**だと腹落ちします。RDKitは発展扱いなので、無くても計算済みの列で全く問題ありません。


## DEEP DIVE：リークしやすい特徴量と、賢い選択

強力だが**リークしやすい**特徴量の代表が**target encoding**（カテゴリを目的変数の平均で置き換える）です。
やり方を誤ると、第6回で学んだリークを自ら仕込むことになります。安全なやり方を身につけます。


### target encoding：全データ平均は「リーク」、OOFなら安全

「系列ごとの平均収率」を特徴量にしたいとします。**全データの平均**で作ると、各行の答えが自分の特徴量に
混ざりリークします。正しくは、第6回の交差検証と同じ発想で、**その行を含まない分割の平均**で作ります
（OOF＝out-of-fold）。両者でMAEを比べ、リークが楽観を生むことを確かめます。


In [ ]:
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import Ridge

def oof_target_encode(frame, col, target, n_splits=5, seed=42):
    "分割の内側で平均を学習するリーク安全なtarget encoding。"
    encoded = pd.Series(index=frame.index, dtype=float)
    global_mean = frame[target].mean()
    for tr, va in KFold(n_splits, shuffle=True, random_state=seed).split(frame):
        means = frame.iloc[tr].groupby(col)[target].mean()
        encoded.iloc[va] = frame.iloc[va][col].map(means).fillna(global_mean).to_numpy()
    return encoded

leaky = df["scaffold_group"].map(df.groupby("scaffold_group")["yield_pct"].mean())
safe = oof_target_encode(df, "scaffold_group", "yield_pct")
num_cols = ["temperature_c", "concentration_m", "logp"]
X_num = df[num_cols].fillna(df[num_cols].median())
for label, enc in {"リークあり(全データ平均)": leaky, "OOF(安全)": safe}.items():
    feats = X_num.assign(scaffold_te=enc.to_numpy())
    scores = cross_val_score(Ridge(), feats, df["yield_pct"], cv=5, scoring="neg_mean_absolute_error")
    print(f"{label}: MAE={-scores.mean():.3f}")
print("リークありは楽観的に見えることがある。実運用の性能はOOFに近い。")


### 出力の読み方

「リークあり」のMAEが「OOF」より**小さく（良く）見える**ことがあります。しかしそれは幻——本番では
その行の答えは手に入りません。**実運用の実力はOFFの側**。強力な特徴量ほど、作り方のリークに注意します。


### RFECV：交差検証つきで特徴量を絞り込む

`RFECV`は、重要度の低い特徴量を1つずつ削りながら交差検証し、**性能が最も良くなる特徴量の組**を
自動で選びます。人手の取捨選択より客観的です。


In [ ]:
from sklearn.feature_selection import RFECV
from sklearn.ensemble import RandomForestRegressor

rfe_data = engineered[added].fillna(engineered[added].median())
rfecv = RFECV(RandomForestRegressor(n_estimators=100, random_state=42), cv=5, scoring="neg_mean_absolute_error", min_features_to_select=2)
rfecv.fit(rfe_data, engineered["yield_pct"])
print("選ばれた特徴量数:", rfecv.n_features_)
pd.DataFrame({"特徴量": added, "残す": rfecv.support_, "順位": rfecv.ranking_}).sort_values("順位")


### 出力の読み方

`残す=True`が採用された特徴量、`順位=1`が最重要グループです。**残った特徴量を化学的に解釈**して
みましょう——自作の`temperature_distance`が残っていれば、知識を式にした狙いが的中したということ。
選択も交差検証の内側で行うことで、選びすぎ（過学習）を避けています。


## よくある誤り

- 意味を説明できない特徴量を大量追加する
- 目的変数由来の値を全データで作って特徴量にする
- 追加前後で分割やモデルも変える

## SELF-STUDY（任意・30〜60分）

- 自作KFold target encodingの有無でMAEを比較する
- RFECVで残った特徴量と、化学的な解釈を突き合わせる

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. その特徴量はいつ計算できるか
2. target encodingでリークを防ぐ手順は何か
3. 特徴量選択も交差検証の内側で行う理由は何か

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
